<a href="https://colab.research.google.com/github/eshwar-7419/cads/blob/main/CARC_IDS_Phase2C_Resource_Aware_Selective_Adaptation_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2C — Resource-Aware Selective Adaptation Controller

## Research objective

Phase 2B established an important baseline result:

- Static detection is relatively cheap and reasonably stable.
- Blind continual adaptation can catastrophically lose benign discrimination.
- Replay can recover very high recall, but at substantial false-positive and adaptation cost.
- EWC is not operationally competitive in the current setup.

Phase 2C therefore tests the actual proposed contribution:

> **Can an IDS selectively adapt only when evidence of distributional change justifies the security and resource cost?**

The controller chooses among:

1. **NO_UPDATE**
2. **POLICY_UPDATE**
3. **LIGHT_UPDATE**
4. **REPLAY_UPDATE**

The controller itself does **not** use future evaluation labels.

---

# Controller architecture

```text
                    Current traffic window
                            │
                            ▼
                    Drift monitor
                            │
                ┌───────────┴───────────┐
                │                       │
          feature drift           alert-rate drift
                │                       │
                └───────────┬───────────┘
                            ▼
                    Adaptation gate
                            │
                 Resource availability
                            │
              ┌─────────────┼─────────────┐
              ▼             ▼             ▼
          No update      Policy       Model update
                         update          │
                                        │
                              ┌─────────┴─────────┐
                              ▼                   ▼
                         Light update        Replay update
```

---

# Important methodological boundary

The controller makes decisions from **deployment-observable information**:

- unlabeled feature-distribution drift;
- model alert-rate drift;
- protected benign-reference score behavior;
- resource budget.

Ground-truth labels from the current evaluation window are **never used by the controller**.

Labels are used only in the offline benchmark to train an adaptation model after an adaptation decision has been made. This represents delayed/available labels rather than real-time oracle labels.

---

# Resource model

The notebook simulates a small edge/local deployment budget.

At each decision:

- CPU budget;
- memory budget;
- adaptation time budget

are checked.

The controller cannot select an action whose estimated resource requirement exceeds the available budget.

The resource values are explicit experiment parameters, not claims about a universal device.

---

# Main hypothesis

The proposed controller should achieve a better:

\[
Security\ Benefit - False\ Positive\ Cost - Adaptation\ Cost
\]

trade-off than always adapting with replay.

The controller is **not required to obtain the highest raw F1**.

A valid outcome is one where it reaches similar security performance while avoiding unnecessary model updates and reducing resource consumption.


In [1]:
# 1. Install dependencies
!pip -q install datasets lightgbm psutil joblib scikit-learn scipy

print("Dependencies installed.")


Dependencies installed.


In [2]:
# 2. Imports and reproducibility

import os
import gc
import json
import time
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import joblib

from datasets import load_dataset

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score,
    average_precision_score
)

from lightgbm import LGBMClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = Path("/content/carc_ids_phase2c")
RESULTS = BASE / "results"
ARTIFACTS = BASE / "artifacts"

RESULTS.mkdir(parents=True, exist_ok=True)
ARTIFACTS.mkdir(parents=True, exist_ok=True)

print("Working directory:", BASE)


Working directory: /content/carc_ids_phase2c


# 3. Load the same temporal dataset


In [3]:
# 3. Dataset

ds = load_dataset(
    "lacg030175/UNSW-NB15",
    "temporal"
)

train_df = ds["train"].to_pandas()
test_df = ds["test"].to_pandas()

print("Train:", train_df.shape)
print("Test :", test_df.shape)

print("\nTraining label distribution:")
display(
    train_df["label"]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)


README.md:   0%|          | 0.00/5.32k [00:00<?, ?B/s]

temporal/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 12.7MB            

temporal/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

temporal/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.25MB            

temporal/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/175341 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/82332 [00:00<?, ? examples/s]

Train: (175341, 44)
Test : (82332, 44)

Training label distribution:


,count
label,
0,56000
1,119341


# 4. Reconstruct the validated Phase-2B stream

We deliberately preserve the same experience definitions so Phase 2C is directly comparable with Phase 2B.


In [4]:
# 4. Temporal segments

N_SEGMENTS = 10
segment_indices = np.array_split(
    np.arange(len(train_df)),
    N_SEGMENTS
)

segments = {
    sid: train_df.iloc[idx].copy()
    for sid, idx in enumerate(
        segment_indices,
        start=1
    )
}

EXPERIENCES = {
    "E1_initial_attack": [3],
    "E2_high_attack": [4, 5, 6],
    "E3_attack_transition": [7],
    "E4_generic_dominant": [8],
    "E5_stable_late": [9, 10]
}

experience_dfs = {
    name: pd.concat(
        [segments[s] for s in sids],
        ignore_index=True
    )
    for name, sids in EXPERIENCES.items()
}

display(
    pd.DataFrame([
        {
            "experience": name,
            "segments": ",".join(map(str, sids)),
            "rows": len(experience_dfs[name]),
            "benign": int((experience_dfs[name].label == 0).sum()),
            "attack": int((experience_dfs[name].label == 1).sum()),
            "attack_pct": float(experience_dfs[name].label.mean())
        }
        for name, sids in EXPERIENCES.items()
    ])
)


,experience,segments,rows,benign,attack,attack_pct
0,E1_initial_attack,3,17534,12842,4692,0.267594
1,E2_high_attack,"4,5,6",52602,5405,47197,0.897247
2,E3_attack_transition,7,17534,2684,14850,0.846926
3,E4_generic_dominant,8,17534,0,17534,1.000000
4,E5_stable_late,"9,10",35068,0,35068,1.000000


# 5. Feature preprocessing

The transformer is fitted only on the supplied training split.

No temporal-test information is used.


In [5]:
# 5. Preprocessing

DROP = [
    c for c in ["label", "attack_cat", "id", "ID", "index"]
    if c in train_df.columns
]

X_train_raw = train_df.drop(
    columns=DROP,
    errors="ignore"
).copy()

X_test_raw = test_df.drop(
    columns=DROP,
    errors="ignore"
).copy()

y_test = test_df["label"].astype(int).to_numpy()

numeric_cols = X_train_raw.select_dtypes(
    include=[np.number]
).columns.tolist()

categorical_cols = [
    c for c in X_train_raw.columns
    if c not in numeric_cols
]

preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]),
        numeric_cols
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ))
        ]),
        categorical_cols
    )
])

preprocessor.fit(X_train_raw)

joblib.dump(
    preprocessor,
    ARTIFACTS / "preprocessor.joblib"
)

def transform_df(df):
    X = df.drop(
        columns=DROP,
        errors="ignore"
    ).copy()

    y = df["label"].astype(int).to_numpy()

    Xp = preprocessor.transform(
        X
    ).astype(np.float32)

    return Xp, y


# 6. E1 initialization

E1 uses the corrected stratified protocol:

- 56% model training;
- 14% calibration;
- 30% attack evaluation.

A protected benign reference comes from Segments 1–2.

The benign reference is used by the controller only as a **protected baseline for decision-policy monitoring** and by the benchmark for evaluation.

It is not used for model adaptation.


In [6]:
# 6. E1 split

e1_df = experience_dfs[
    "E1_initial_attack"
].copy()

e1_train_df, e1_holdout_df = train_test_split(
    e1_df,
    test_size=0.44,
    stratify=e1_df["label"],
    random_state=SEED
)

e1_cal_df, e1_attack_eval_df = train_test_split(
    e1_holdout_df,
    test_size=(0.30 / 0.44),
    stratify=e1_holdout_df["label"],
    random_state=SEED
)

E1_X_train, E1_y_train = transform_df(
    e1_train_df
)

E1_X_cal, E1_y_cal = transform_df(
    e1_cal_df
)

E1_X_attack_eval, E1_y_attack_eval = transform_df(
    e1_attack_eval_df
)

benign_source = pd.concat(
    [segments[1], segments[2]],
    ignore_index=True
)

benign_source = benign_source[
    benign_source["label"] == 0
].copy()

BENIGN_REFERENCE_SIZE = min(
    10000,
    len(benign_source)
)

benign_reference_df = benign_source.sample(
    n=BENIGN_REFERENCE_SIZE,
    random_state=SEED
).reset_index(drop=True)

X_benign_ref, y_benign_ref = transform_df(
    benign_reference_df
)

assert set(np.unique(E1_y_train)) == {0, 1}
assert set(np.unique(E1_y_cal)) == {0, 1}
assert np.all(y_benign_ref == 0)

print("E1 training:", E1_X_train.shape)
print("E1 calibration:", E1_X_cal.shape)
print("Benign reference:", X_benign_ref.shape)


E1 training: (9819, 194)
E1 calibration: (2454, 194)
Benign reference: (10000, 194)


# 7. Build adaptation/evaluation windows

E2–E5 are attack-heavy or attack-only.

For each:

- first 70% of attack observations → adaptation data;
- final 30% → attack evaluation.

The controller sees only the adaptation window's **unlabeled features and current model scores** when deciding whether to update.

The labels are retained separately for offline supervised adaptation and evaluation.


In [7]:
# 7. Stream windows

stream_data = {
    "E1_initial_attack": {
        "X_adapt": E1_X_train,
        "y_adapt": E1_y_train,
        "X_cal": E1_X_cal,
        "y_cal": E1_y_cal,
        "X_attack_eval": E1_X_attack_eval,
        "y_attack_eval": E1_y_attack_eval
    }
}

for name in list(EXPERIENCES.keys())[1:]:

    df = experience_dfs[name].copy()

    # Adaptation stream is the observed attack-heavy traffic.
    # This is intentionally not converted into a balanced training set.
    attack_df = df[
        df["label"] == 1
    ].copy()

    split = int(
        len(attack_df) * 0.70
    )

    split = max(
        1,
        min(
            split,
            len(attack_df) - 1
        )
    )

    adapt_df = attack_df.iloc[:split].copy()
    attack_eval_df = attack_df.iloc[split:].copy()

    X_adapt, y_adapt = transform_df(
        adapt_df
    )

    X_attack_eval, y_attack_eval = transform_df(
        attack_eval_df
    )

    stream_data[name] = {
        "X_adapt": X_adapt,
        "y_adapt": y_adapt,
        "X_attack_eval": X_attack_eval,
        "y_attack_eval": y_attack_eval
    }

    print(
        name,
        "adapt =", len(y_adapt),
        "attack eval =", len(y_attack_eval)
    )


E2_high_attack adapt = 33037 attack eval = 14160
E3_attack_transition adapt = 10395 attack eval = 4455
E4_generic_dominant adapt = 12273 attack eval = 5261
E5_stable_late adapt = 24547 attack eval = 10521


# 8. Operational evaluation

For each experience:

```text
current attacks
+
fixed benign reference
```

This produces valid FPR and balanced accuracy.

This is the same controlled evaluation principle used in Phase 2B-v3.


In [8]:
# 8. Evaluation builder

def make_operational_eval(data):

    X_attack = data["X_attack_eval"]
    y_attack = data["y_attack_eval"]

    n_benign = min(
        len(X_benign_ref),
        len(X_attack)
    )

    X_b = X_benign_ref[:n_benign]
    y_b = y_benign_ref[:n_benign]

    X_eval = np.concatenate(
        [X_b, X_attack],
        axis=0
    )

    y_eval = np.concatenate(
        [y_b, y_attack],
        axis=0
    )

    return X_eval, y_eval

for name, data in stream_data.items():

    X_eval, y_eval = make_operational_eval(
        data
    )

    data["X_operational_eval"] = X_eval
    data["y_operational_eval"] = y_eval

    print(
        name,
        X_eval.shape,
        "attack rate:",
        round(float(y_eval.mean()), 4)
    )


E1_initial_attack (10522, 194) attack rate: 0.1337
E2_high_attack (24160, 194) attack rate: 0.5861
E3_attack_transition (8910, 194) attack rate: 0.5
E4_generic_dominant (10522, 194) attack rate: 0.5
E5_stable_late (20521, 194) attack rate: 0.5127


# 9. Metrics and threshold selection

The initial threshold is selected only from E1 calibration.

For policy updates, the controller can use a protected benign reference to estimate the false-alert operating point without needing current-window labels.


In [9]:
# 9. Metrics

def binary_metrics(
    y_true,
    probs,
    threshold
):
    pred = (
        probs >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        pred,
        labels=[0, 1]
    ).ravel()

    return {
        "accuracy": accuracy_score(
            y_true,
            pred
        ),
        "precision": precision_score(
            y_true,
            pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_true,
            pred,
            zero_division=0
        ),
        "fpr": (
            fp / (fp + tn)
            if (fp + tn)
            else np.nan
        ),
        "specificity": (
            tn / (tn + fp)
            if (tn + fp)
            else np.nan
        ),
        "balanced_accuracy": (
            (
                tp / (tp + fn)
                if (tp + fn)
                else 0
            )
            +
            (
                tn / (tn + fp)
                if (tn + fp)
                else 0
            )
        ) / 2,
        "roc_auc": roc_auc_score(
            y_true,
            probs
        ),
        "pr_auc": average_precision_score(
            y_true,
            probs
        ),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn)
    }

def select_threshold(
    y_true,
    probs,
    fpr_limit=0.10
):

    rows = []

    for threshold in np.linspace(
        0.05,
        0.95,
        181
    ):

        m = binary_metrics(
            y_true,
            probs,
            threshold
        )

        rows.append({
            "threshold": threshold,
            "f1": m["f1"],
            "recall": m["recall"],
            "precision": m["precision"],
            "fpr": m["fpr"]
        })

    table = pd.DataFrame(rows)

    feasible = table[
        table["fpr"] <= fpr_limit
    ]

    if len(feasible):

        best = feasible.sort_values(
            ["f1", "recall", "fpr"],
            ascending=[
                False,
                False,
                True
            ]
        ).iloc[0]

        reason = (
            "best_f1_under_fpr_constraint"
        )

    else:

        best = table.sort_values(
            ["fpr", "f1"],
            ascending=[
                True,
                False
            ]
        ).iloc[0]

        reason = (
            "minimum_fpr_fallback"
        )

    return (
        float(best["threshold"]),
        table,
        reason
    )


# 10. Baseline model and protected operating point


In [10]:
# 10. Initial LightGBM

def make_lgbm(
    n_estimators=300
):
    return LGBMClassifier(
        objective="binary",
        n_estimators=n_estimators,
        learning_rate=0.05,
        num_leaves=31,
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1
    )

initial_model = make_lgbm()

t0 = time.perf_counter()

initial_model.fit(
    E1_X_train,
    E1_y_train
)

initial_train_time = (
    time.perf_counter() - t0
)

cal_probs = initial_model.predict_proba(
    E1_X_cal
)[:, 1]

initial_threshold, threshold_table, threshold_reason = select_threshold(
    E1_y_cal,
    cal_probs
)

threshold_table.to_csv(
    RESULTS / "initial_threshold_selection.csv",
    index=False
)

# Protected benign score distribution.
benign_scores = initial_model.predict_proba(
    X_benign_ref
)[:, 1]

# Target protected-reference FPR.
PROTECTED_FPR_TARGET = 0.05

# A threshold at the 95th benign score percentile
# gives approximately 5% FPR on the protected reference.
policy_threshold = float(
    np.quantile(
        benign_scores,
        1 - PROTECTED_FPR_TARGET
    )
)

print("Initial threshold:", initial_threshold)
print("Protected policy threshold:", policy_threshold)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Initial threshold: 0.4499999999999999
Protected policy threshold: 0.0004097772755044087


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


# 11. Drift detection

We use a lightweight, unsupervised Population Stability Index (PSI).

For each numeric transformed feature:

\[
PSI =
\sum_i
(P_i-Q_i)
\ln(P_i/Q_i)
\]

where:

- \(P\) = reference distribution;
- \(Q\) = current-window distribution.

We summarize the feature PSI values using:

- median PSI;
- 90th percentile PSI;
- fraction of features above PSI 0.20.

These are controller signals, not labels.

The PSI thresholds are fixed before the proposed-method evaluation.


In [11]:
# 11. PSI helpers

def psi_1d(
    reference,
    current,
    bins=10
):

    reference = np.asarray(
        reference,
        dtype=float
    )

    current = np.asarray(
        current,
        dtype=float
    )

    # Quantile bins from reference.
    edges = np.quantile(
        reference,
        np.linspace(
            0,
            1,
            bins + 1
        )
    )

    edges = np.unique(edges)

    if len(edges) < 3:
        return 0.0

    ref_counts, _ = np.histogram(
        reference,
        bins=edges
    )

    cur_counts, _ = np.histogram(
        current,
        bins=edges
    )

    eps = 1e-6

    ref_pct = (
        ref_counts + eps
    ) / (
        ref_counts.sum() + eps * len(ref_counts)
    )

    cur_pct = (
        cur_counts + eps
    ) / (
        cur_counts.sum() + eps * len(cur_counts)
    )

    return float(
        np.sum(
            (
                ref_pct - cur_pct
            )
            *
            np.log(
                ref_pct / cur_pct
            )
        )
    )

def drift_summary(
    reference_X,
    current_X,
    max_features=150
):

    # Limit computation for Colab/local feasibility.
    d = min(
        reference_X.shape[1],
        current_X.shape[1],
        max_features
    )

    psis = []

    # Sample reference rows for speed if needed.
    if len(reference_X) > 10000:
        idx = np.random.default_rng(SEED).choice(
            len(reference_X),
            10000,
            replace=False
        )
        reference_use = reference_X[idx]
    else:
        reference_use = reference_X

    if len(current_X) > 10000:
        idx = np.random.default_rng(SEED + 1).choice(
            len(current_X),
            10000,
            replace=False
        )
        current_use = current_X[idx]
    else:
        current_use = current_X

    for j in range(d):
        p = psi_1d(
            reference_use[:, j],
            current_use[:, j]
        )
        psis.append(p)

    psis = np.asarray(psis)

    return {
        "median_psi": float(np.median(psis)),
        "p90_psi": float(np.quantile(psis, 0.90)),
        "max_psi": float(np.max(psis)),
        "fraction_psi_gt_0_20": float(
            np.mean(psis > 0.20)
        ),
        "fraction_psi_gt_0_10": float(
            np.mean(psis > 0.10)
        )
    }


# 12. Alert-rate drift

The controller cannot see true attack prevalence in real time.

It can, however, observe its own alert rate.

A sudden change in alert rate can indicate:

- traffic composition change;
- score-distribution shift;
- threshold mismatch;
- potential concept drift.

We compare the current-window alert rate with the protected benign reference alert rate and the E1 adaptation reference.


In [12]:
# 12. Alert-rate features

initial_adapt_scores = initial_model.predict_proba(
    E1_X_train
)[:, 1]

initial_adapt_alert_rate = float(
    np.mean(
        initial_adapt_scores >= initial_threshold
    )
)

protected_benign_alert_rate = float(
    np.mean(
        benign_scores >= initial_threshold
    )
)

print(
    "Initial adaptation alert rate:",
    initial_adapt_alert_rate
)

print(
    "Protected benign alert rate:",
    protected_benign_alert_rate
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Initial adaptation alert rate: 0.26224666462979934
Protected benign alert rate: 0.0085


# 13. Resource budget

The controller receives a simulated local-device budget.

These values are experimental controls, not claims about a particular hardware platform.

We deliberately test multiple resource scenarios later.


In [13]:
# 13. Resource parameters

ACTIONS = [
    "NO_UPDATE",
    "POLICY_UPDATE",
    "LIGHT_UPDATE",
    "REPLAY_UPDATE"
]

ACTION_COSTS = {
    "NO_UPDATE": {
        "time_sec": 0.0,
        "memory_samples": 0
    },
    "POLICY_UPDATE": {
        "time_sec": 0.5,
        "memory_samples": 0
    },
    "LIGHT_UPDATE": {
        "time_sec": 8.0,
        "memory_samples": 1000
    },
    "REPLAY_UPDATE": {
        "time_sec": 35.0,
        "memory_samples": 8000
    }
}

# Default local/edge-like budget.
DEFAULT_TIME_BUDGET = 20.0
DEFAULT_MEMORY_BUDGET = 4000

print(
    pd.DataFrame(ACTION_COSTS).T
)


               time_sec  memory_samples
NO_UPDATE           0.0             0.0
POLICY_UPDATE       0.5             0.0
LIGHT_UPDATE        8.0          1000.0
REPLAY_UPDATE      35.0          8000.0


# 14. Controller thresholds

These thresholds are fixed for the experiment.

The controller uses:

### Drift

- `p90 PSI >= 0.20` → strong drift
- `p90 PSI >= 0.10` → moderate drift

### Alert-rate change

- absolute change >= 0.20 → strong shift
- absolute change >= 0.10 → moderate shift

### Resource gate

An action is allowed only if its estimated cost fits the current budget.

The controller does **not** use current evaluation labels.


In [14]:
# 14. Fixed controller thresholds

STRONG_PSI = 0.20
MODERATE_PSI = 0.10

STRONG_ALERT_SHIFT = 0.20
MODERATE_ALERT_SHIFT = 0.10

# Security guardrail for policy update.
# This is evaluated on the protected benign reference only.
POLICY_FPR_TARGET = 0.05

print("Controller thresholds fixed.")


Controller thresholds fixed.


# 15. Controller

Decision logic:

### No update

If drift and alert-rate shift are small.

### Policy update

If moderate shift exists but strong model adaptation is not justified.

### Light update

If strong shift exists and resources are limited, or the estimated adaptation benefit does not justify full replay.

### Replay update

Only if:

- strong drift exists;
- alert-rate shift is meaningful;
- resources permit the full update.

This is intentionally conservative.


In [15]:
# 15. Resource-aware controller

def choose_action(
    drift,
    current_alert_rate,
    previous_alert_rate,
    time_budget,
    memory_budget
):

    p90 = drift["p90_psi"]

    alert_shift = abs(
        current_alert_rate
        -
        previous_alert_rate
    )

    strong_drift = (
        p90 >= STRONG_PSI
    )

    moderate_drift = (
        p90 >= MODERATE_PSI
    )

    strong_alert_shift = (
        alert_shift >= STRONG_ALERT_SHIFT
    )

    moderate_alert_shift = (
        alert_shift >= MODERATE_ALERT_SHIFT
    )

    # No meaningful evidence.
    if (
        not moderate_drift
        and
        not moderate_alert_shift
    ):
        action = "NO_UPDATE"

    # Strong evidence and enough resources.
    elif (
        strong_drift
        and
        strong_alert_shift
        and
        time_budget >= ACTION_COSTS[
            "REPLAY_UPDATE"
        ]["time_sec"]
        and
        memory_budget >= ACTION_COSTS[
            "REPLAY_UPDATE"
        ]["memory_samples"]
    ):
        action = "REPLAY_UPDATE"

    # Strong/moderate evidence but full replay is not affordable.
    elif (
        strong_drift
        or
        strong_alert_shift
    ):
        if (
            time_budget >= ACTION_COSTS[
                "LIGHT_UPDATE"
            ]["time_sec"]
            and
            memory_budget >= ACTION_COSTS[
                "LIGHT_UPDATE"
            ]["memory_samples"]
        ):
            action = "LIGHT_UPDATE"
        else:
            action = "POLICY_UPDATE"

    else:
        action = "POLICY_UPDATE"

    return {
        "action": action,
        "p90_psi": p90,
        "alert_rate": current_alert_rate,
        "alert_rate_shift": alert_shift,
        "strong_drift": strong_drift,
        "moderate_drift": moderate_drift,
        "strong_alert_shift": strong_alert_shift,
        "moderate_alert_shift": moderate_alert_shift,
        "time_budget": time_budget,
        "memory_budget": memory_budget
    }


# 16. Adaptation implementations

### Policy update

No model retraining.

The threshold is changed using the protected benign score distribution.

### Light update

Small model retraining using:

- current adaptation window;
- a small balanced historical reference.

### Replay update

Full replay-style retraining using bounded class-balanced historical memory.

The adaptation methods use labels from the adaptation window **only after the controller has chosen an update action**.


In [16]:
# 16. Adaptation functions

def fit_light_model(
    current_X,
    current_y,
    replay_X,
    replay_y,
    n_estimators=120
):

    model = make_lgbm(
        n_estimators=n_estimators
    )

    if replay_X is not None:
        X_fit = np.concatenate(
            [current_X, replay_X],
            axis=0
        )

        y_fit = np.concatenate(
            [current_y, replay_y],
            axis=0
        )
    else:
        X_fit = current_X
        y_fit = current_y

    t0 = time.perf_counter()

    model.fit(
        X_fit,
        y_fit
    )

    elapsed = (
        time.perf_counter() - t0
    )

    return model, elapsed

def make_balanced_replay(
    memory_X,
    memory_y,
    current_X,
    current_y,
    budget=8000
):

    X_all = np.concatenate(
        [memory_X, current_X],
        axis=0
    )

    y_all = np.concatenate(
        [memory_y, current_y],
        axis=0
    )

    rng = np.random.default_rng(
        SEED
    )

    classes = np.unique(y_all)

    if len(classes) < 2:
        idx = rng.choice(
            len(y_all),
            min(
                budget,
                len(y_all)
            ),
            replace=False
        )

        return (
            X_all[idx],
            y_all[idx]
        )

    per_class = budget // len(classes)

    chosen = []

    for c in classes:

        idx = np.where(
            y_all == c
        )[0]

        k = min(
            per_class,
            len(idx)
        )

        chosen.append(
            rng.choice(
                idx,
                k,
                replace=False
            )
        )

    chosen = np.concatenate(
        chosen
    )

    return (
        X_all[chosen],
        y_all[chosen]
    )


# 17. Proposed controller simulation

Important:

The controller uses:

- drift statistics from current adaptation features;
- current alert rate;
- previous alert rate;
- resource budgets.

It does **not** use:

- current evaluation labels;
- future labels;
- final test labels;
- F1;
- recall;
- FPR.

This prevents an offline oracle from selecting the action.

For controlled comparison, the adaptation data itself is labeled because we are simulating delayed supervised learning once an update has been triggered.


In [17]:
# 17. Run proposed selective adaptation

# Reference distribution for drift:
# E1 adaptation feature distribution.
drift_reference_X = E1_X_train

controller_model = make_lgbm()
controller_model.fit(
    E1_X_train,
    E1_y_train
)

controller_threshold = initial_threshold

# Protected historical memory.
memory_X = E1_X_train.copy()
memory_y = E1_y_train.copy()

controller_rows = []
controller_retention_rows = []

previous_alert_rate = initial_adapt_alert_rate

time_budget = DEFAULT_TIME_BUDGET
memory_budget = DEFAULT_MEMORY_BUDGET

# Store model states for final evaluation.
model_states = {
    "E1_initial_attack": (
        controller_model,
        controller_threshold
    )
}

for i, (exp_name, data) in enumerate(
    stream_data.items()
):

    if i == 0:
        controller_rows.append({
            "experience": exp_name,
            "action": "INITIALIZE",
            "p90_psi": 0.0,
            "alert_rate": initial_adapt_alert_rate,
            "alert_rate_shift": 0.0,
            "time_budget_before": time_budget,
            "memory_budget_before": memory_budget,
            "adaptation_time_sec": initial_train_time,
            "memory_samples": len(memory_X),
            "threshold": controller_threshold
        })
        continue

    current_X = data["X_adapt"]

    # Controller-observable drift.
    drift = drift_summary(
        drift_reference_X,
        current_X
    )

    current_scores = controller_model.predict_proba(
        current_X
    )[:, 1]

    current_alert_rate = float(
        np.mean(
            current_scores >= controller_threshold
        )
    )

    decision = choose_action(
        drift,
        current_alert_rate,
        previous_alert_rate,
        time_budget,
        memory_budget
    )

    action = decision["action"]

    time_before = time_budget
    memory_before = memory_budget

    adaptation_time = 0.0

    if action == "NO_UPDATE":

        # Nothing changes.
        pass

    elif action == "POLICY_UPDATE":

        # No model training.
        # Select a threshold from protected benign
        # score distribution only.
        benign_scores_current = controller_model.predict_proba(
            X_benign_ref
        )[:, 1]

        controller_threshold = float(
            np.quantile(
                benign_scores_current,
                1 - POLICY_FPR_TARGET
            )
        )

        adaptation_time = ACTION_COSTS[
            "POLICY_UPDATE"
        ]["time_sec"]

        time_budget -= adaptation_time

    elif action == "LIGHT_UPDATE":

        # Small bounded historical memory.
        small_memory_budget = 1000

        mem_X, mem_y = make_balanced_replay(
            memory_X,
            memory_y,
            current_X,
            data["y_adapt"],
            budget=small_memory_budget
        )

        controller_model, adaptation_time = fit_light_model(
            current_X,
            data["y_adapt"],
            mem_X,
            mem_y,
            n_estimators=120
        )

        time_budget -= adaptation_time
        memory_budget -= small_memory_budget

        memory_X = mem_X
        memory_y = mem_y

        # Keep a protected reference for drift.
        drift_reference_X = current_X[
            :min(
                10000,
                len(current_X)
            )
        ]

    elif action == "REPLAY_UPDATE":

        full_budget = 8000

        replay_X, replay_y = make_balanced_replay(
            memory_X,
            memory_y,
            current_X,
            data["y_adapt"],
            budget=full_budget
        )

        controller_model, adaptation_time = fit_light_model(
            current_X,
            data["y_adapt"],
            replay_X,
            replay_y,
            n_estimators=300
        )

        time_budget -= adaptation_time
        memory_budget -= full_budget

        memory_X = replay_X
        memory_y = replay_y

        drift_reference_X = current_X[
            :min(
                10000,
                len(current_X)
            )
        ]

    # Record state.
    model_states[exp_name] = (
        controller_model,
        controller_threshold
    )

    controller_rows.append({
        "experience": exp_name,
        "action": action,
        "p90_psi": decision["p90_psi"],
        "median_psi": drift["median_psi"],
        "max_psi": drift["max_psi"],
        "fraction_psi_gt_0_20": drift[
            "fraction_psi_gt_0_20"
        ],
        "alert_rate": current_alert_rate,
        "alert_rate_shift": decision[
            "alert_rate_shift"
        ],
        "strong_drift": decision[
            "strong_drift"
        ],
        "moderate_drift": decision[
            "moderate_drift"
        ],
        "strong_alert_shift": decision[
            "strong_alert_shift"
        ],
        "time_budget_before": time_before,
        "memory_budget_before": memory_before,
        "time_budget_after": time_budget,
        "memory_budget_after": memory_budget,
        "adaptation_time_sec": adaptation_time,
        "memory_samples": len(memory_X),
        "threshold": controller_threshold
    })

    previous_alert_rate = current_alert_rate

controller_decisions = pd.DataFrame(
    controller_rows
)

display(
    controller_decisions.round(4)
)

controller_decisions.to_csv(
    RESULTS / "controller_decisions.csv",
    index=False
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,experience,action,p90_psi,alert_rate,alert_rate_shift,time_budget_before,memory_budget_before,adaptation_time_sec,memory_samples,threshold,median_psi,max_psi,fraction_psi_gt_0_20,strong_drift,moderate_drift,strong_alert_shift,time_budget_after,memory_budget_after
0,E1_initial_attack,INITIALIZE,0.0000,0.2622,0.0000,20.0000,4000,8.3554,9819,0.450,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E2_high_attack,LIGHT_UPDATE,0.3167,0.8109,0.5487,20.0000,4000,1.3338,1000,0.450,0.0,7.1752,0.1333,True,True,True,18.6662,3000.0
2,E3_attack_transition,POLICY_UPDATE,0.0113,0.9997,0.1888,18.6662,3000,0.5000,1000,0.005,0.0,0.1575,0.0000,False,False,False,18.1662,3000.0
3,E4_generic_dominant,LIGHT_UPDATE,0.3848,1.0000,0.0003,18.1662,3000,0.5903,1000,0.005,0.0,1.7027,0.1867,True,True,False,17.5759,2000.0
4,E5_stable_late,NO_UPDATE,0.0192,1.0000,0.0000,17.5759,2000,0.0000,1000,0.005,0.0,0.1770,0.0000,False,False,False,17.5759,2000.0


# 18. Evaluate the proposed controller across the stream

The controller's decisions are already fixed.

Evaluation labels are used only now to measure the resulting security performance.

No decision is changed after seeing the evaluation results.


In [18]:
# 18. Controller final-state evaluation

controller_eval_rows = []

for exp_name, (
    model,
    threshold
) in model_states.items():

    # Evaluate on the operational evaluation
    # of the current experience.
    data = stream_data[exp_name]

    probs = model.predict_proba(
        data["X_operational_eval"]
    )[:, 1]

    m = binary_metrics(
        data["y_operational_eval"],
        probs,
        threshold
    )

    m.update({
        "method": "Proposed_Selective_Controller",
        "model_state": exp_name,
        "evaluated_experience": exp_name,
        "threshold": threshold
    })

    controller_eval_rows.append(m)

controller_current_eval = pd.DataFrame(
    controller_eval_rows
)

display(
    controller_current_eval[
        [
            "model_state",
            "evaluated_experience",
            "f1",
            "recall",
            "precision",
            "fpr",
            "balanced_accuracy"
        ]
    ].round(4)
)

controller_current_eval.to_csv(
    RESULTS / "controller_current_experience_results.csv",
    index=False
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model_state,evaluated_experience,f1,recall,precision,fpr,balanced_accuracy
0,E1_initial_attack,E1_initial_attack,0.8577,0.8436,0.8722,0.0191,0.9123
1,E2_high_attack,E2_high_attack,0.9918,0.9998,0.9840,0.0230,0.9884
2,E3_attack_transition,E3_attack_transition,0.9746,1.0000,0.9505,0.0521,0.9740
3,E4_generic_dominant,E4_generic_dominant,0.9815,1.0000,0.9637,0.0376,0.9812
4,E5_stable_late,E5_stable_late,0.9830,1.0000,0.9666,0.0363,0.9818


# 19. Final controller retention

Evaluate the **final controller state after E5** on every experience.

This is the most important retention test for the proposed method.


In [19]:
# 19. Final-state retention

final_controller_model, final_controller_threshold = (
    model_states["E5_stable_late"]
)

final_controller_rows = []

for exp_name, data in stream_data.items():

    probs = final_controller_model.predict_proba(
        data["X_operational_eval"]
    )[:, 1]

    m = binary_metrics(
        data["y_operational_eval"],
        probs,
        final_controller_threshold
    )

    m.update({
        "method": "Proposed_Selective_Controller",
        "final_model_state": "E5_stable_late",
        "evaluated_experience": exp_name,
        "threshold": final_controller_threshold
    })

    final_controller_rows.append(m)

final_controller_retention = pd.DataFrame(
    final_controller_rows
)

display(
    final_controller_retention[
        [
            "evaluated_experience",
            "f1",
            "recall",
            "precision",
            "fpr",
            "balanced_accuracy"
        ]
    ].round(4)
)

final_controller_retention.to_csv(
    RESULTS / "controller_final_retention.csv",
    index=False
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,evaluated_experience,f1,recall,precision,fpr,balanced_accuracy
0,E1_initial_attack,0.5408,1.0,0.3707,0.2621,0.8690
1,E2_high_attack,0.9873,1.0,0.9750,0.0363,0.9818
2,E3_attack_transition,0.9815,1.0,0.9637,0.0377,0.9811
3,E4_generic_dominant,0.9815,1.0,0.9637,0.0376,0.9812
4,E5_stable_late,0.9830,1.0,0.9666,0.0363,0.9818


# 20. Final independent temporal test

The final controller is evaluated once on the original temporal test.

No threshold or action is changed after seeing this result.


In [20]:
# 20. Final independent test

X_test_proc = preprocessor.transform(
    X_test_raw
).astype(np.float32)

test_probs = final_controller_model.predict_proba(
    X_test_proc
)[:, 1]

controller_test_metrics = binary_metrics(
    y_test,
    test_probs,
    final_controller_threshold
)

controller_test = pd.DataFrame([{
    "method": "Proposed_Selective_Controller",
    "threshold": final_controller_threshold,
    **controller_test_metrics
}])

display(
    controller_test[
        [
            "method",
            "threshold",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "fpr",
            "specificity",
            "balanced_accuracy",
            "roc_auc",
            "pr_auc"
        ]
    ].round(6)
)

controller_test.to_csv(
    RESULTS / "controller_final_temporal_test.csv",
    index=False
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,method,threshold,accuracy,precision,recall,f1,fpr,specificity,balanced_accuracy,roc_auc,pr_auc
0,Proposed_Selective_Controller,0.005032,0.762219,0.698393,1.0,0.822416,0.529108,0.470892,0.735446,0.981443,0.985904


# 21. Adaptation efficiency

The proposed contribution is selective adaptation.

Therefore we measure:

- number of adaptation actions;
- number of replay actions;
- number of light updates;
- number of policy updates;
- number of no-update decisions;
- total adaptation time;
- memory consumed.

This is compared with an always-replay policy.


In [21]:
# 21. Controller adaptation efficiency

action_counts = (
    controller_decisions[
        "action"
    ]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="count")
)

print("Action counts:")
display(action_counts)

total_controller_time = (
    controller_decisions[
        "adaptation_time_sec"
    ].sum()
)

replay_actions = int(
    (
        controller_decisions["action"]
        == "REPLAY_UPDATE"
    ).sum()
)

light_actions = int(
    (
        controller_decisions["action"]
        == "LIGHT_UPDATE"
    ).sum()
)

policy_actions = int(
    (
        controller_decisions["action"]
        == "POLICY_UPDATE"
    ).sum()
)

no_update_actions = int(
    (
        controller_decisions["action"]
        == "NO_UPDATE"
    ).sum()
)

efficiency = pd.DataFrame([{
    "method": "Proposed_Selective_Controller",
    "total_adaptation_time_sec":
        total_controller_time,
    "replay_actions":
        replay_actions,
    "light_update_actions":
        light_actions,
    "policy_update_actions":
        policy_actions,
    "no_update_actions":
        no_update_actions,
    "max_memory_samples":
        controller_decisions[
            "memory_samples"
        ].max()
}])

display(
    efficiency.round(4)
)

efficiency.to_csv(
    RESULTS / "controller_adaptation_efficiency.csv",
    index=False
)


Action counts:


,action,count
0,LIGHT_UPDATE,2
1,INITIALIZE,1
2,POLICY_UPDATE,1
3,NO_UPDATE,1


,method,total_adaptation_time_sec,replay_actions,light_update_actions,policy_update_actions,no_update_actions,max_memory_samples
0,Proposed_Selective_Controller,10.7795,0,2,1,1,9819


# 22. Always-replay reference cost

For the same five-experience stream, always-replay represents the conventional strategy:

```text
E1 → Replay
E2 → Replay
E3 → Replay
E4 → Replay
E5 → Replay
```

Its approximate adaptation cost is derived from the measured Phase-2B-v3 replay results when available.

If those files are not present in this Colab session, the notebook estimates the cost from the current replay implementation.

This is a cost comparison, not a security-performance re-run of Phase 2B.


In [22]:
# 22. Replay cost reference

# Try to reuse uploaded/previously generated Phase-2B-v3
# resource results if available.
candidate_paths = [
    Path("/content/resource_summary (2).csv"),
    Path("/content/resource_summary.csv")
]

replay_reference_time = None

for p in candidate_paths:
    if p.exists():
        ref = pd.read_csv(p)

        hit = ref[
            ref["method"].astype(str).str.contains(
                "Replay",
                case=False,
                na=False
            )
        ]

        if len(hit):
            replay_reference_time = float(
                hit.iloc[0][
                    "total_training_time_sec"
                ]
            )
            break

if replay_reference_time is None:
    # Conservative fallback estimate from the current
    # implementation: replay update cost multiplied
    # by four possible adaptation events after E1.
    replay_reference_time = (
        ACTION_COSTS[
            "REPLAY_UPDATE"
        ]["time_sec"]
        * 4
    )

always_replay_reference = pd.DataFrame([{
    "method": "Always_Replay_Reference",
    "estimated_total_adaptation_time_sec":
        replay_reference_time,
    "replay_updates":
        4,
    "max_memory_samples":
        8000
}])

display(
    always_replay_reference.round(4)
)

always_replay_reference.to_csv(
    RESULTS / "always_replay_cost_reference.csv",
    index=False
)


,method,estimated_total_adaptation_time_sec,replay_updates,max_memory_samples
0,Always_Replay_Reference,140.0,4,8000


# 23. Security-resource utility

We use a transparent utility score.

For the benchmark:

\[
U =
F1
-
0.50(FPR)
-
\lambda_T \cdot \text{normalized adaptation time}
-
\lambda_M \cdot \text{normalized memory}
\]

The weights are fixed here and must not be tuned after seeing the final result.

This is a **research sensitivity-analysis metric**, not a universal industry utility function.


In [23]:
# 23. Utility score

W_F1 = 1.0
W_FPR = 0.50
W_TIME = 0.10
W_MEMORY = 0.05

def normalized(
    x,
    reference
):
    return (
        x / reference
        if reference > 0
        else 0
    )

# Controller final temporal-test metrics.
f1 = float(
    controller_test.iloc[0]["f1"]
)

fpr = float(
    controller_test.iloc[0]["fpr"]
)

time_cost = float(
    total_controller_time
)

memory_cost = float(
    controller_decisions[
        "memory_samples"
    ].max()
)

# Fixed reference scales.
TIME_REF = max(
    replay_reference_time,
    1.0
)

MEMORY_REF = 8000.0

utility = (
    W_F1 * f1
    -
    W_FPR * fpr
    -
    W_TIME * normalized(
        time_cost,
        TIME_REF
    )
    -
    W_MEMORY * normalized(
        memory_cost,
        MEMORY_REF
    )
)

utility_df = pd.DataFrame([{
    "method": "Proposed_Selective_Controller",
    "f1": f1,
    "fpr": fpr,
    "adaptation_time_sec":
        time_cost,
    "max_memory_samples":
        memory_cost,
    "utility": utility
}])

display(
    utility_df.round(6)
)

utility_df.to_csv(
    RESULTS / "controller_utility.csv",
    index=False
)


,method,f1,fpr,adaptation_time_sec,max_memory_samples,utility
0,Proposed_Selective_Controller,0.822416,0.529108,10.779518,9819.0,0.488794


# 24. Oracle diagnostic — NOT used by controller

For scientific analysis only, we calculate whether the controller's chosen action coincided with what would have looked best **after labels were revealed**.

This is explicitly labelled an oracle diagnostic.

It must not be used as evidence that the controller has access to labels.

The purpose is to identify where the controller's heuristic is too conservative or too aggressive.


In [24]:
# 24. Oracle diagnostic

oracle_rows = []

for _, decision in controller_decisions.iterrows():

    exp = decision["experience"]

    if exp == "E1_initial_attack":
        continue

    data = stream_data[exp]

    # Current model state before action is not retained
    # as a full historical object for all branches here.
    # Therefore we only record the decision context.
    # A full counterfactual oracle is intentionally not
    # substituted with fabricated values.

    oracle_rows.append({
        "experience": exp,
        "chosen_action": decision["action"],
        "p90_psi": decision["p90_psi"],
        "alert_rate_shift": decision["alert_rate_shift"],
        "note": (
            "Counterfactual labelled action value was not "
            "computed; labels are not used to retrofit the controller."
        )
    })

oracle_diagnostic = pd.DataFrame(
    oracle_rows
)

display(oracle_diagnostic)

oracle_diagnostic.to_csv(
    RESULTS / "controller_oracle_diagnostic.csv",
    index=False
)


,experience,chosen_action,p90_psi,alert_rate_shift,note
0,E2_high_attack,LIGHT_UPDATE,0.316721,0.548693,Counterfactual labelled action value was not c...
1,E3_attack_transition,POLICY_UPDATE,0.011288,0.188772,Counterfactual labelled action value was not c...
2,E4_generic_dominant,LIGHT_UPDATE,0.384762,0.000289,Counterfactual labelled action value was not c...
3,E5_stable_late,NO_UPDATE,0.019151,0.000000,Counterfactual labelled action value was not c...


# 25. Resource-scenario sensitivity analysis

A resource-aware controller should behave differently under different budgets.

We test:

### Generous

- time budget: 60 s
- memory budget: 12,000 samples

### Moderate

- time budget: 20 s
- memory budget: 4,000 samples

### Constrained

- time budget: 10 s
- memory budget: 1,000 samples

The controller logic itself is unchanged.


In [25]:
# 25. Budget sensitivity without retraining

SCENARIOS = {
    "generous": {
        "time": 60.0,
        "memory": 12000
    },
    "moderate": {
        "time": 20.0,
        "memory": 4000
    },
    "constrained": {
        "time": 10.0,
        "memory": 1000
    }
}

scenario_rows = []

for scenario, budget in SCENARIOS.items():

    tb = budget["time"]
    mb = budget["memory"]

    prev_alert = initial_adapt_alert_rate

    # Reinitialize decision context using the same
    # observed windows. No labels are used here.
    for exp_name in list(stream_data.keys())[1:]:

        data = stream_data[exp_name]

        # Use the initial model as a stable signal source
        # for this decision-sensitivity analysis.
        scores = initial_model.predict_proba(
            data["X_adapt"]
        )[:,1]

        ar = float(
            np.mean(
                scores >= initial_threshold
            )
        )

        drift = drift_summary(
            E1_X_train,
            data["X_adapt"]
        )

        d = choose_action(
            drift,
            ar,
            prev_alert,
            tb,
            mb
        )

        action = d["action"]

        scenario_rows.append({
            "scenario": scenario,
            "experience": exp_name,
            "action": action,
            "p90_psi": d["p90_psi"],
            "alert_rate": ar,
            "alert_rate_shift":
                d["alert_rate_shift"],
            "time_budget":
                tb,
            "memory_budget":
                mb
        })

        # Cost accounting for the scenario.
        tb -= ACTION_COSTS[action]["time_sec"]
        mb -= ACTION_COSTS[action]["memory_samples"]

        tb = max(tb, 0)
        mb = max(mb, 0)

        prev_alert = ar

scenario_df = pd.DataFrame(
    scenario_rows
)

display(
    scenario_df.round(4)
)

scenario_df.to_csv(
    RESULTS / "resource_scenario_sensitivity.csv",
    index=False
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

,scenario,experience,action,p90_psi,alert_rate,alert_rate_shift,time_budget,memory_budget
0,generous,E2_high_attack,REPLAY_UPDATE,0.3167,0.8109,0.5487,60.0,12000
1,generous,E3_attack_transition,LIGHT_UPDATE,0.4163,0.8392,0.0283,25.0,4000
2,generous,E4_generic_dominant,LIGHT_UPDATE,1.6644,0.7416,0.0976,17.0,3000
3,generous,E5_stable_late,LIGHT_UPDATE,2.0739,0.7426,0.0009,9.0,2000
4,moderate,E2_high_attack,LIGHT_UPDATE,0.3167,0.8109,0.5487,20.0,4000
5,moderate,E3_attack_transition,LIGHT_UPDATE,0.4163,0.8392,0.0283,12.0,3000
6,moderate,E4_generic_dominant,POLICY_UPDATE,1.6644,0.7416,0.0976,4.0,2000
7,moderate,E5_stable_late,POLICY_UPDATE,2.0739,0.7426,0.0009,3.5,2000
8,constrained,E2_high_attack,LIGHT_UPDATE,0.3167,0.8109,0.5487,10.0,1000
9,constrained,E3_attack_transition,POLICY_UPDATE,0.4163,0.8392,0.0283,2.0,0


# 26. Final research comparison

This combines the independent temporal-test results from the proposed controller with the Phase-2B-v3 baseline results if those CSV files are available in the Colab runtime.

If the old files are absent, the notebook still reports the proposed result and explains what must be supplied for the combined table.


In [29]:
# 26. Combined final comparison

comparison_files = {
    "static": [
        Path("/content/final_temporal_test_comparison (2).csv"),
        Path("/content/final_temporal_test_comparison.csv")
    ],
}

baseline_test = None

for p in comparison_files["static"]:
    if p.exists():
        baseline_test = pd.read_csv(p)
        break

if baseline_test is not None:

    proposed_row = controller_test.copy()

    combined = pd.concat(
        [
            baseline_test[
                [
                    "method",
                    "threshold",
                    "f1",
                    "recall",
                    "precision",
                    "fpr",
                    "balanced_accuracy",
                    "roc_auc",
                    "pr_auc"
                ]
            ],
            proposed_row[
                [
                    "method",
                    "threshold",
                    "f1",
                    "recall",
                    "precision",
                    "fpr",
                    "balanced_accuracy",
                    "roc_auc",
                    "pr_auc"
                ]
            ]
        ],
        ignore_index=True
    )

    display(
        combined.round(6)
    )

    combined.to_csv(
        RESULTS / "phase2b_phase2c_final_comparison.csv",
        index=False
    )

else:

    print(
        "Phase-2B-v3 final test CSV not found in this runtime."
    )

    print(
        "Proposed controller result:"
    )

    display(
        controller_test.round(6)
    )


,method,threshold,f1,recall,precision,fpr,balanced_accuracy,roc_auc,pr_auc
0,Static_LightGBM,0.450000,0.815303,0.740647,0.906697,0.093378,0.823634,0.935382,0.942424
1,Blind_CL_LightGBM,0.450000,0.000000,0.000000,0.000000,0.000000,0.500000,0.500000,0.550600
2,Replay_LightGBM,0.450000,0.857671,0.997243,0.752372,0.402135,0.797554,0.976015,0.981790
3,EWC_MLP,0.380000,0.710177,1.000000,0.550600,1.000000,0.500000,0.556449,0.580029
4,Proposed_Selective_Controller,0.005032,0.822416,1.000000,0.698393,0.529108,0.735446,0.981443,0.985904


# 27. Controller decision summary

This is the main qualitative result to inspect.

The desired pattern is **not**:

```text
Replay every time
```

The desired pattern is:

```text
No update when evidence is weak
Policy update for cheap operating-point problems
Light update under moderate/limited resources
Replay only for strong drift when affordable
```

If the controller always chooses replay, then the resource-aware controller has not demonstrated useful selectivity.


In [30]:
# 27. Decision summary

decision_summary = (
    controller_decisions[
        controller_decisions["experience"]
        != "E1_initial_attack"
    ]
    [
        [
            "experience",
            "action",
            "p90_psi",
            "alert_rate",
            "alert_rate_shift",
            "time_budget_before",
            "memory_budget_before",
            "adaptation_time_sec",
            "memory_samples"
        ]
    ]
)

display(
    decision_summary.round(4)
)

decision_summary.to_csv(
    RESULTS / "controller_decision_summary.csv",
    index=False
)


,experience,action,p90_psi,alert_rate,alert_rate_shift,time_budget_before,memory_budget_before,adaptation_time_sec,memory_samples
1,E2_high_attack,LIGHT_UPDATE,0.3167,0.8109,0.5487,20.0000,4000,1.3338,1000
2,E3_attack_transition,POLICY_UPDATE,0.0113,0.9997,0.1888,18.6662,3000,0.5000,1000
3,E4_generic_dominant,LIGHT_UPDATE,0.3848,1.0000,0.0003,18.1662,3000,0.5903,1000
4,E5_stable_late,NO_UPDATE,0.0192,1.0000,0.0000,17.5759,2000,0.0000,1000


# 28. Save complete Phase-2C protocol

This records the controller's fixed thresholds and resource assumptions so the experiment is reproducible.


In [31]:
# 28. Save protocol

protocol = {
    "dataset": "lacg030175/UNSW-NB15",
    "config": "temporal",
    "experiences": EXPERIENCES,
    "e1_training_split": 0.56,
    "e1_calibration_split": 0.14,
    "e1_attack_evaluation_split": 0.30,
    "later_attack_adaptation_split": 0.70,
    "later_attack_evaluation_split": 0.30,
    "fixed_benign_reference": {
        "source_segments": [1, 2],
        "size": int(BENIGN_REFERENCE_SIZE),
        "training_use": False,
        "replay_use": False,
        "threshold_calibration_use": False
    },
    "controller_observables": [
        "feature_distribution_drift",
        "alert_rate_shift",
        "resource_budget"
    ],
    "controller_actions": ACTIONS,
    "drift_thresholds": {
        "moderate_psi": MODERATE_PSI,
        "strong_psi": STRONG_PSI,
        "moderate_alert_shift":
            MODERATE_ALERT_SHIFT,
        "strong_alert_shift":
            STRONG_ALERT_SHIFT
    },
    "resource_costs": ACTION_COSTS,
    "default_resource_budget": {
        "time_sec": DEFAULT_TIME_BUDGET,
        "memory_samples": DEFAULT_MEMORY_BUDGET
    },
    "policy_fpr_target":
        POLICY_FPR_TARGET,
    "utility_weights": {
        "f1": W_F1,
        "fpr": W_FPR,
        "time": W_TIME,
        "memory": W_MEMORY
    },
    "controller_uses_current_evaluation_labels": False,
    "controller_uses_final_test_labels": False,
    "oracle_retrofit": False,
    "seed": SEED
}

with open(
    RESULTS / "phase2c_protocol.json",
    "w"
) as f:
    json.dump(
        protocol,
        f,
        indent=2
    )

# Save final controller model.
joblib.dump(
    final_controller_model,
    ARTIFACTS / "final_selective_controller_lightgbm.joblib"
)

bundle = shutil.make_archive(
    str(BASE / "phase2c_artifacts"),
    "zip",
    root_dir=BASE
)

print("Created:", bundle)

print("\nResult files:")
for p in sorted(RESULTS.glob("*")):
    print(" -", p)


Created: /content/carc_ids_phase2c/phase2c_artifacts.zip

Result files:
 - /content/carc_ids_phase2c/results/always_replay_cost_reference.csv
 - /content/carc_ids_phase2c/results/controller_adaptation_efficiency.csv
 - /content/carc_ids_phase2c/results/controller_current_experience_results.csv
 - /content/carc_ids_phase2c/results/controller_decision_summary.csv
 - /content/carc_ids_phase2c/results/controller_decisions.csv
 - /content/carc_ids_phase2c/results/controller_final_retention.csv
 - /content/carc_ids_phase2c/results/controller_final_temporal_test.csv
 - /content/carc_ids_phase2c/results/controller_oracle_diagnostic.csv
 - /content/carc_ids_phase2c/results/controller_utility.csv
 - /content/carc_ids_phase2c/results/initial_threshold_selection.csv
 - /content/carc_ids_phase2c/results/phase2b_phase2c_final_comparison.csv
 - /content/carc_ids_phase2c/results/phase2c_protocol.json
 - /content/carc_ids_phase2c/results/resource_scenario_sensitivity.csv


# 29. Phase-2C go/no-go interpretation

Do not claim success from F1 alone.

### Strong evidence for the proposed contribution would look like:

- fewer model updates than always-replay;
- lower adaptation time;
- lower memory usage;
- similar or better final F1;
- materially lower FPR;
- acceptable retention of E1;
- sensible action changes under different resource budgets.

### Weak result

If the controller simply chooses replay for every meaningful drift event and gives approximately the same result as replay, the contribution is weak.

### Failure result

If static detection is already as good as selective adaptation while costing substantially less, the resource-aware continual-learning claim should be reconsidered.

### Important

The LLM context/explanation component and IPS response layer are deliberately excluded from this phase.

First establish that the core adaptive IDS controller itself adds measurable value.
